<a href="https://colab.research.google.com/github/MamoMGD1/ISE302-DataMining-GroupProject/blob/main/shared/01_core_numeric.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Core Numeric & Date Columns (Team 1)

## Goal of this notebook
This notebook processes **11 core numeric and date columns** from the raw car listing dataset. Our objective is to transform raw web-scraped strings into a clean, model-ready numeric format.

### Our Methodology:
1. **Extraction & Unit Stripping** — Removing currency (TL), distance (km), and volume (lt) units.
2. **Temporal Feature Engineering** — Transforming `İlan Tarihi` into a continuous integer (days passed) to capture the effect of listing age on price.
3. **Robust Imputation** — Utilizing **Median Imputation** across all features. We chose the median over the mean because numeric car data (Price/Mileage) is heavily right-skewed; the median is a robust measure of central tendency that is not biased by high-end luxury vehicle outliers.
4. **Validation** — Strict numeric casting and null-checks to ensure compatibility with the Phase 1 final merge.

## Columns Handled
`Fiyat`, `Yıl`, `Kilometre`, `İlan Tarihi`, `Ortalama Kasko`, `Ortalama Trafik Sigortası`, `Üretim Yılı (İlk/Son)`, `Silindir Sayısı`, `Koltuk Sayısı`, `Bagaj Hacmi`, `Yakıt Deposu`

## Output
`df_team1` — A fully numeric DataFrame with 3,424 rows.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")

# Load the raw dataset directly from GitHub
RAW_URL = "https://raw.githubusercontent.com/MamoMGD1/ISE302-DataMining-GroupProject/main/data/raw_dataset.csv"
df_full = pd.read_csv(RAW_URL)

# Isolate Team 1's assigned columns
MY_COLUMNS = [
    'Fiyat', 'Yıl', 'Kilometre', 'İlan Tarihi',
    'Ortalama Kasko', 'Ortalama Trafik Sigortası',
    'Üretim Yılı (İlk/Son)', 'Silindir Sayısı',
    'Koltuk Sayısı', 'Bagaj Hacmi', 'Yakıt Deposu'
]
df = df_full[MY_COLUMNS].copy()

print(f"✅ Subset Loaded. Shape: {df.shape}")
df.head(3)

## Step 1 — Pre-Cleaning Exploratory Analysis
We first visualize the missing value density. Columns like `Ortalama Kasko` and `Bagaj Hacmi` typically show higher null counts in scraped data. Identifying this distribution justifies our use of **Median Imputation** in the next step to ensure we do not lose 30%+ of our data rows.

In [ ]:
# Calculate null percentages
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

# Visualize
plt.figure(figsize=(12, 5))
sns.barplot(x=null_pct.index, y=null_pct.values, palette='magma')
plt.title('Missing Value Percentage by Column', fontsize=14)
plt.ylabel('% Missing', fontsize=12)
plt.xticks(rotation=45)
plt.show()

print("Exact Null Counts:")
print(df.isnull().sum())

## Step 2 — Data Cleaning & Feature Engineering
**What we are doing:**
* **String Cleaning:** We strip non-numeric characters (TL, km, dots, spaces).
* **Turkish Formatting Fix:** We replace commas (`,`) with dots (`.`) for decimal columns like `Bagaj Hacmi` to ensure correct float conversion.
* **Temporal Logic:** `İlan Tarihi` is converted to "Days Since Minimum Date". This turns a categorical date string into a quantitative distance-in-time feature.
* **Generation Extraction:** For `Üretim Yılı (İlk/Son)`, we extract only the start year (the first 4-digit match) as it marks the technological beginning of that car's model generation.

In [ ]:
# 1. Clean Fiyat (The Target Variable)
df['Fiyat'] = df['Fiyat'].astype(str).str.replace('TL', '', regex=False).str.replace('.', '', regex=False).str.strip().astype(int)

# 2. Clean Yıl, Silindir, Koltuk
for col in ['Yıl', 'Silindir Sayısı', 'Koltuk Sayısı']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median()).astype(int)

# 3. Clean Kilometre & Insurance (Strip units + Handle Median)
for col in ['Kilometre', 'Ortalama Kasko', 'Ortalama Trafik Sigortası']:
    df[col] = (df[col].astype(str)
               .str.replace(' km', '', regex=False)
               .str.replace('TL', '', regex=False)
               .str.replace('.', '', regex=False)
               .str.strip()
               .replace('nan', np.nan))
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median()).astype(int)

# 4. Process İlan Tarihi (Conversion to linear time)
df['İlan Tarihi'] = pd.to_datetime(df['İlan Tarihi'], dayfirst=True, errors='coerce')
min_date = df['İlan Tarihi'].min()
df['İlan Tarihi'] = (df['İlan Tarihi'] - min_date).dt.days
df['İlan Tarihi'] = df['İlan Tarihi'].fillna(df['İlan Tarihi'].median()).astype(int)

# 5. Extract Start Year from Production Range
df['Üretim Yılı (İlk/Son)'] = df['Üretim Yılı (İlk/Son)'].astype(str).str.extract(r'(\d{4})')[0]
df['Üretim Yılı (İlk/Son)'] = pd.to_numeric(df['Üretim Yılı (İlk/Son)'], errors='coerce')
df['Üretim Yılı (İlk/Son)'] = df['Üretim Yılı (İlk/Son)'].fillna(df['Üretim Yılı (İlk/Son)'].median()).astype(int)

# 6. Clean Bagaj & Depo (Turkish decimal fix included)
for col in ['Bagaj Hacmi', 'Yakıt Deposu']:
    df[col] = (df[col].astype(str)
               .str.replace(' lt', '', regex=False)
               .str.replace(',', '.', regex=False) # Fix decimal commas
               .str.strip()
               .replace('nan', np.nan))
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median()).astype(int)

print("✅ Step 2 Transformation Completed.")

## Step 3 — Post-Cleaning Distribution Analysis
We visualize the histograms of key features to confirm data integrity. As observed, features like `Kilometre` and `Fiyat` show a heavy right-skew. This confirms that **Median Imputation** was the scientifically correct choice, as the mean would have been significantly skewed by the expensive/high-mileage outliers visible in these plots.

In [ ]:
# Visualize distributions to check for imputation bias
cols_to_plot = ['Fiyat', 'Kilometre', 'Yıl', 'Bagaj Hacmi']
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for i, col in enumerate(cols_to_plot):
    sns.histplot(df[col], kde=True, ax=axes[i], color='teal')
    axes[i].set_title(f'Distribution of {col}', fontsize=12)

plt.tight_layout()
plt.show()

## Step 4 — Integrity Verification
Before completing Phase 1 for Team 1, we run an automated check to ensure:
1. No missing values remain.
2. Every column has been successfully converted to a numeric type (int or float).

In [ ]:
df_team1 = df.copy()

# Final Integrity Assertions
assert df_team1.isnull().sum().sum() == 0, "❌ Error: Missing values detected!"
assert df_team1.select_dtypes(exclude=['number']).shape[1] == 0, "❌ Error: Non-numeric columns detected!"

print(f"✅ Team 1 Final Output Ready.")
print(f"Shape: {df_team1.shape}")
print("-" * 40)
display(df_team1.info())
display(df_team1.head())